# Qwen Q8 Chat on RunPod (chat-only)
Prepare the environment first with `python scripts/pod_run.py all` (isolated venv, no system pip), then pick the kernel **Python (qwen-venv)**.

Open this notebook from the Pod's Jupyter (connect from the RunPod console → Connect). It can be started from the `notebooks/` folder or the repository root. Run cells 1 → 5 in order. Nothing secret is stored here; the Gradio login is typed at runtime (hidden input).

- Cell 4 sends **one short request with an explicit output cap** (64 tokens). That is a smoke check, not a performance measurement.
- Cell 4b (optional, run only after Cell 4 passed and you decided to) measures warm runs separately. Performance criteria are evaluated only from those; otherwise they stay `skipped`.
- **Stopping the app here does not stop the Pod.** Stop the Pod from outside: `python scripts/pod.py stop <pod_id>` (REST v2) or the console.

In [ ]:
# Cell 1: repo root, isolated-kernel check, pinned upstream source
# The environment itself (venv + hash-locked install + pip check) is prepared by `python scripts/pod_run.py venv`,
# NOT here: installing into the Pod's system Python breaks on OS packages (PyGObject/pycairo) that are not ours.
import os, pathlib, subprocess, sys

def find_repo_root(start=None):
    """Works from the repo root or from notebooks/ (or QMC_REPO_ROOT)."""
    first = [pathlib.Path(os.environ["QMC_REPO_ROOT"])] if os.environ.get("QMC_REPO_ROOT") else []
    cwd = pathlib.Path(start or os.getcwd()).resolve()
    for p in [*first, cwd, *cwd.parents]:
        if (p / "qmc_runpod" / "pins.py").is_file() and (p / "requirements-runpod.lock.txt").is_file():
            return p
    raise RuntimeError("repository root not found: open this notebook inside the cloned repo or set QMC_REPO_ROOT")

REPO_DIR = find_repo_root()
# --- end root resolution ---
sys.path.insert(0, str(REPO_DIR))
from qmc_runpod import envsetup, layout, pins, provenance, report
if not envsetup.running_in_venv():
    raise RuntimeError("Select the kernel 'Python (qwen-venv)' (Kernel > Change kernel). If it is missing, run: python scripts/pod_run.py venv")
CHECKS = report.checks_template()          # every check starts as "skipped"; only evidence changes it
OPS_COMMIT, OPS_CLEAN = provenance.git_head(REPO_DIR), provenance.tree_clean(REPO_DIR)
report.mark(CHECKS, "ops_commit_clean", OPS_CLEAN)
print("ops commit", OPS_COMMIT[:12], "clean" if OPS_CLEAN else "DIRTY (stop and check)")
P = layout.paths(); P["source"].mkdir(parents=True, exist_ok=True)
SRC = P["source"] / "qwen-multimodal-colab"
if not (SRC / ".git").exists():
    subprocess.run(["git", "clone", "--filter=blob:none", pins.UPSTREAM_REPO, str(SRC)], check=True)
subprocess.run(["git", "-C", str(SRC), "fetch", "origin", pins.UPSTREAM_SHA], check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "-q", pins.UPSTREAM_SHA], check=True)
head = subprocess.run(["git", "-C", str(SRC), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
blob = subprocess.run(["git", "-C", str(SRC), "rev-parse", "HEAD:Qwen-Q8-Chat-Colab.ipynb"], capture_output=True, text=True, check=True).stdout.strip()
report.mark(CHECKS, "upstream_pin", head == pins.UPSTREAM_SHA and blob == pins.UPSTREAM_NOTEBOOK_BLOB)
assert CHECKS["upstream_pin"] == "pass", "upstream pin mismatch"
LOCK = REPO_DIR / "requirements-runpod.lock.txt"
report.mark(CHECKS, "lock_installed", subprocess.run([sys.executable, "-m", "pip", "check"]).returncode == 0)   # inside the venv only
sys.path.insert(0, str(SRC / "src"))
print("upstream", head[:10], "lock sha256", provenance.file_sha256(LOCK)[:12], "python", sys.version.split()[0], "venv", sys.prefix)

In [ ]:
# Cell 2: non-secret env, llama-server at the pinned commit (built here), models at pinned revisions (SHA256-verified)
import time
from qmc_runpod import llama, models
ENV = layout.apply_env(ctx=8192, web_search="off")   # 8k first; 32768 is a separate later run
existing = list(layout.paths()["llama_bin"].glob("*/llama-server"))
t0 = time.monotonic()
SERVER = llama.install()
BUILD_SECONDS = round(time.monotonic() - t0, 1)
BUILD = {"cuda_archs": SERVER.parent.name.split("-sm")[-1].split("-py")[0], "runtime_tag": llama.runtime_tag(),
         "build_type": "Release", "build_seconds": BUILD_SECONDS, "built_this_run": not existing}
report.mark(CHECKS, "llama_built", SERVER.exists())
MODEL_PATHS, MODEL_RECORDS = models.fetch_pinned(layout.paths()["hf_cache"])
report.mark(CHECKS, "models_verified", all(r["sha256_verified"] for r in MODEL_RECORDS))
print(SERVER, BUILD, [r["sha256_verified"] for r in MODEL_RECORDS])

In [ ]:
# Cell 3: start the chat-only UI. Needs BOTH login values (hidden input); refuses to start otherwise.
import getpass
os.environ["QMC_AUTH_USER"] = getpass.getpass("Gradio user: ")
os.environ["QMC_AUTH_PASSWORD"] = getpass.getpass("Gradio password: ")
from qmc_runpod import launch
APP = launch.launch(model_paths=MODEL_PATHS)
report.mark(CHECKS, "app_launch", True)
report.mark(CHECKS, "auth_enforced", launch.check_auth_enforced(APP.cfg.server_port))
print(f"https://{os.environ.get('RUNPOD_POD_ID', '<pod-id>')}-{APP.cfg.server_port}.proxy.runpod.net/", "auth enforced:", CHECKS["auth_enforced"])

In [ ]:
# Cell 4: ONE short request, output capped at 64 tokens. Smoke check only (this request is the cold one).
from qmc_runpod import smoke
LOAD = smoke.load_model(APP)                 # model start, timed separately
FIRST = smoke.first_response(APP)            # max_tokens=64, shown in the result
GPU = smoke.gpu_memory_mib()
report.mark(CHECKS, "first_response", (not FIRST["error"]) and FIRST["stream_deltas"] > 0)
print(LOAD, FIRST, GPU)

In [ ]:
# Cell 4b (OPTIONAL): warm runs, only after Cell 4 passed and you decided to. Cold and warm stay separate.
PROMPTS = ["日本の四季を50字で説明して。", "1から10までの和は？", "Pythonでリストを逆順にする方法は？",
           "富士山の高さは？", "挨拶を一言。", "Gitのcommitとpushの違いは？",
           "味噌汁の基本の作り方を3行で。", "TCPとUDPの違いを一言で。", "今日の気分を一言で。", "素数とは？"]
assert CHECKS["first_response"] == "pass", "run Cell 4 first and fix any failure"
WARM = smoke.warm_runs(APP, PROMPTS)         # max_tokens=256 each
WARM_SUMMARY = smoke.summarize(WARM)
CRITERIA = smoke.evaluate_criteria(WARM)     # "not_evaluated" unless 10 clean warm runs
report.mark(CHECKS, "warm_runs", WARM_SUMMARY["ok_runs"] == len(PROMPTS))
report.mark(CHECKS, "perf_criteria", None if CRITERIA["verdict"] == "not_evaluated" else CRITERIA["verdict"] == "pass")
print(WARM_SUMMARY, CRITERIA)

In [ ]:
# Cell 5: allowlisted report + bundle. Download the .tar.gz from Jupyter; then stop the Pod from outside.
import datetime, gradio
from qmc_runpod import trial
metrics = {"load_s": LOAD["load_s"], "first_max_tokens": FIRST["max_tokens"], "first_total_s": FIRST["total_s"],
           "first_stream_deltas": FIRST["stream_deltas"], "first_finish_reason": FIRST["finish_reason"]}
if FIRST["first_delta_s"] is not None:
    metrics["first_first_delta_s"] = FIRST["first_delta_s"]
if GPU.get("used_mib") is not None:
    metrics["vram_used_mib"] = GPU["used_mib"]
if "WARM_SUMMARY" in globals():
    metrics.update({"warm_" + k: v for k, v in WARM_SUMMARY.items()})
rep = trial.build_report(
    trial_id="t" + datetime.datetime.now().strftime("%Y%m%d-%H%M"), repo_root=REPO_DIR, lock_path=LOCK,
    model_records=MODEL_RECORDS, gpu=GPU, build=BUILD, checks=CHECKS, metrics=metrics,
    server_version=llama.server_version(SERVER), gradio_version=gradio.__version__,
    settings={"ctx": int(ENV["QMC_CHAT_CTX"]), "thinking": False, "web_search": "off", "asr_device": "cpu",
              "asr_model": "small", "tts": False, "share": False, "chat_only": True},
    notes=["report_exported is confirmed outside the Pod after read-back", "perf criteria need 10 warm runs"])
path = report.write_report(rep, layout.paths()["runs"])
bundle = layout.paths()["runs"] / (path.stem + ".tar.gz")
report.make_bundle([path], bundle)
print(bundle)

In [ ]:
# Cell 6: stop the app and the model process. This does NOT stop or delete the Pod.
launch.stop_app(APP)